# 01 - Data download and cleaning

The raw Appliances Energy Prediction file is sampled every 10 minutes. This notebook
loads it, checks it for the problems that would invalidate a time-series analysis
(gaps, duplicates, irregular spacing), and resamples it to the hourly frequency used
throughout the project.

The README for the assignment allows resampling to hourly data to keep SARIMAX
estimation manageable. That is what we do, and the consequences are discussed at the
end of the notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting

## Load the raw 10-minute data

`data.load_raw` parses the timestamp, sorts the index, drops the two random columns
`rv1` and `rv2` that the original authors added as noise variables, and coerces
everything else to numeric.

In [ ]:
raw = data.load_raw()

print(f"Rows: {len(raw)}")
print(f"Columns: {len(raw.columns)}")
print(f"Period: {raw.index.min()} to {raw.index.max()}")
raw.head()

## Data quality checks

Before doing anything else, confirm the index is well behaved.

In [ ]:
spacing = raw.index.to_series().diff().value_counts()

print("Duplicated timestamps:", raw.index.duplicated().sum())
print("Missing values:", int(raw.isna().sum().sum()))
print("\nSpacing between observations:")
print(spacing)

The index is completely regular: every one of the 19,734 gaps is exactly 10 minutes,
there are no duplicates and no missing values. This is unusually clean for sensor data
and means no imputation is needed at the raw frequency.

## Resample to hourly

Hourly means are used rather than sums, which keeps the target on its original Wh scale
so that the hourly and 10-minute series are directly comparable.

In [ ]:
hourly = data.to_hourly(raw)

print(f"Hourly observations: {len(hourly)}")
print(f"Period: {hourly.index.min()} to {hourly.index.max()}")
print(f"Days: {(hourly.index.max() - hourly.index.min()).days}")

regular = (hourly.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all()
print(f"Strictly regular hourly index: {regular}")
print(f"Missing target values: {int(hourly[config.TARGET].isna().sum())}")

## Effect of resampling on the target

Averaging removes some of the sharpest short-lived spikes. It is worth quantifying that,
because it changes what the models are being asked to predict.

In [ ]:
comparison = pd.DataFrame({
    "10-minute": data.describe_series(raw[config.TARGET]),
    "hourly": data.describe_series(hourly[config.TARGET]),
})

comparison

The maximum falls from 1,080 Wh to 608 Wh and the standard deviation from 103 to 81,
while the mean is essentially unchanged at about 98 Wh. Averaging has removed the most
extreme ten-minute bursts but left the overall level and the strong right skew intact.

This matters for interpretation. A model that forecasts the hourly series well is not
necessarily able to anticipate a two-minute kettle spike, and the assignment's target of
short-term household forecasting is arguably better served at the hourly resolution
anyway, since that is the granularity at which most tariff and scheduling decisions are
made.

## Save the processed dataset

In [ ]:
config.ensure_dirs()
hourly.to_csv(config.HOURLY_CSV)

print(f"Saved to {config.HOURLY_CSV}")
print(f"Shape: {hourly.shape}")

## Train and test split

The assignment recommends the final 14 days as the test period, which is 336 hourly
observations.

In [ ]:
train, test = data.train_test_split(hourly[config.TARGET])

print(f"Train: {train.index.min()} to {train.index.max()}  ({len(train)} obs)")
print(f"Test:  {test.index.min()} to {test.index.max()}  ({len(test)} obs)")
print(f"\nTest share of the sample: {len(test) / len(hourly):.1%}")

The test period covers 13 to 27 May 2016. The training sample is a little over four
months, which is long enough to estimate daily and weekly seasonality but too short to
say anything about annual seasonality. That limitation is carried through the whole
analysis: nothing here can distinguish a May effect from a level shift.